# Селективная фильтрация сейсмического разреза по локальному наклону

Локальная ориентация волнового поля оценивается по структурному тензору. Затем гауссовой весовой функцией усиливаются элементы разреза, ориентация которых близка к заданному углу.

## 1. Подготовка

По умолчанию используется конец куба (последние 700 трасс), где в исходной курсовой анализировался
участок с выраженным нарушением. Угол здесь измеряется **в координатах изображения**; без учёта физического
шага трасс и временного/глубинного шага это не истинный геологический угол падения.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

!pip -q install segyio

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import segyio
from scipy import ndimage

SEGY_PATH = Path('/content/drive/MyDrive/cw_data/TrainingData_Image.segy')
N_TRACES = 700
SMOOTH_SIGMA = 5.0
TARGET_ANGLE_DEG = -30.0
ANGLE_SIGMA_DEG = 12.0

if not SEGY_PATH.exists():
    raise FileNotFoundError(f'SEG-Y не найден: {SEGY_PATH}')

## 2. Загрузка выбранного фрагмента

In [ ]:
def robust_normalize(data: np.ndarray, percentile: float = 99.5) -> np.ndarray:
    """Нормирует амплитуды по устойчивому перцентилю, не меняя знак."""
    scale = np.percentile(np.abs(data), percentile)
    if scale <= np.finfo(np.float32).eps:
        return np.zeros_like(data, dtype=np.float32)
    return np.clip(data / scale, -1.0, 1.0).astype(np.float32)


def load_last_traces(path: Path, n_traces: int) -> np.ndarray:
    """Возвращает массив (sample, trace) для последних n_traces."""
    with segyio.open(str(path), 'r', ignore_geometry=True) as segy:
        total = len(segy.trace)
        count = min(n_traces, total)
        start = total - count
        traces = np.stack([
            np.asarray(segy.trace[i], dtype=np.float32)
            for i in range(start, total)
        ])
    return traces.T


seismic = load_last_traces(SEGY_PATH, N_TRACES)
seismic_norm = robust_normalize(seismic)
print(f'Размер разреза: {seismic_norm.shape[0]} отсчётов × {seismic_norm.shape[1]} трасс')

## 3. Локальная ориентация по структурному тензору

In [ ]:
def compute_local_orientation(data: np.ndarray, sigma: float = 5.0) -> np.ndarray:
    """Оценивает локальную ориентацию структуры в градусах, диапазон [-90, 90)."""
    grad_trace = ndimage.sobel(data, axis=1, mode='reflect')
    grad_time = ndimage.sobel(data, axis=0, mode='reflect')

    j_xx = ndimage.gaussian_filter(grad_trace * grad_trace, sigma=sigma)
    j_tt = ndimage.gaussian_filter(grad_time * grad_time, sigma=sigma)
    j_xt = ndimage.gaussian_filter(grad_trace * grad_time, sigma=sigma)

    angle_rad = 0.5 * np.arctan2(2.0 * j_xt, j_xx - j_tt)
    return np.degrees(angle_rad).astype(np.float32)


orientation_deg = compute_local_orientation(seismic_norm, sigma=SMOOTH_SIGMA)

## 4. Гауссова селекция по углу

In [ ]:
def orientation_difference(angle: np.ndarray, target: float) -> np.ndarray:
    """Минимальная разность ориентаций с учётом периодичности 180°."""
    return (angle - target + 90.0) % 180.0 - 90.0


def gaussian_orientation_filter(
    data: np.ndarray,
    orientation: np.ndarray,
    target_angle: float,
    sigma_angle: float,
) -> tuple[np.ndarray, np.ndarray]:
    delta = orientation_difference(orientation, target_angle)
    weights = np.exp(-0.5 * (delta / sigma_angle) ** 2).astype(np.float32)
    return data * weights, weights


selective_image, angle_weights = gaussian_orientation_filter(
    seismic_norm,
    orientation_deg,
    target_angle=TARGET_ANGLE_DEG,
    sigma_angle=ANGLE_SIGMA_DEG,
)

## 5. Результат

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 6), constrained_layout=True)

axes[0].imshow(seismic_norm, cmap='seismic', aspect='auto', vmin=-0.5, vmax=0.5)
axes[0].set_title('Исходный разрез')

im1 = axes[1].imshow(orientation_deg, cmap='RdBu_r', aspect='auto', vmin=-45, vmax=45)
axes[1].set_title('Локальная ориентация, °')
fig.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(angle_weights, cmap='viridis', aspect='auto', vmin=0, vmax=1)
axes[2].set_title(f'Вес для {TARGET_ANGLE_DEG:g}°')
fig.colorbar(im2, ax=axes[2], shrink=0.8)

axes[3].imshow(selective_image, cmap='seismic', aspect='auto', vmin=-0.5, vmax=0.5)
axes[3].set_title('Селективное изображение')

for ax in axes:
    ax.set_xlabel('Трасса')
    ax.set_ylabel('Временной отсчёт')

plt.show()

## Интерпретация

Метод не «находит разлом» напрямую. Он подчёркивает элементы изображения с выбранной локальной ориентацией.
Это удобно как вспомогательный атрибут: при удачно выбранном угле наклонные элементы нарушения могут стать заметнее.
Результат чувствителен к `SMOOTH_SIGMA`, `TARGET_ANGLE_DEG` и `ANGLE_SIGMA_DEG`, поэтому эти параметры следует
подбирать на нескольких участках, а не только на одном очевидном примере.